# Phase 7 — Classical Hypothesis Testing: Radius Distributions Across Groups

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp, mannwhitneyu, kruskal, chi2_contingency, fisher_exact

from pathlib import Path

def find_data_dir():
    """Locate the project's data/ folder from anywhere inside the project tree."""
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / "data").is_dir():
            return p / "data"
    raise FileNotFoundError("Could not locate the project's data/ folder")

DATA = find_data_dir()

df = pd.read_csv((DATA / "filtered_main_sample.csv"))

df['log_rade'] = np.log10(df['pl_rade'])
GAP_LOCATION = 1.76
df['above_gap'] = (df['pl_rade'] > GAP_LOCATION).astype(int)

print(df.shape)
print(df[['pl_rade', 'log_rade', 'pl_orbper', 'above_gap']].head())

(2587, 89)
   pl_rade  log_rade  pl_orbper  above_gap
0  1.87500  0.273001   0.736547          1
1  2.79000  0.445604  18.859023          1
2  2.57807  0.411295  41.685500          1
3  3.70000  0.568202   5.113400          1
4  1.68135  0.225658   0.853592          0


In [2]:
df['period_group'] = np.where(df['pl_orbper'] <= 10, 'short', 'long')
print(df['period_group'].value_counts())
print()
print(df.groupby('period_group')['pl_rade'].describe())

period_group
short    1415
long     1172
Name: count, dtype: int64

               count      mean       std   min   25%   50%   75%  max
period_group                                                         
long          1172.0  2.335036  0.684538  0.54  1.88  2.34  2.78  4.0
short         1415.0  1.847077  0.739895  0.51  1.28  1.67  2.39  4.0


In [3]:
short_radius = df.loc[df['period_group'] == 'short', 'pl_rade']
long_radius = df.loc[df['period_group'] == 'long', 'pl_rade']

ks_stat, ks_p = ks_2samp(short_radius, long_radius)
print(f"KS test: statistic = {ks_stat:.4f}, p-value = {ks_p:.4e}")

mw_stat, mw_p = mannwhitneyu(short_radius, long_radius, alternative='two-sided')
print(f"Mann-Whitney U test: statistic = {mw_stat:.4f}, p-value = {mw_p:.4e}")

KS test: statistic = 0.3391, p-value = 7.7681e-66
Mann-Whitney U test: statistic = 509419.5000, p-value = 3.8915e-64


## Radius distribution: short-period vs long-period planets

We tested whether radius distributions differ between short-period (≤10 days) and long-period (>10 days) planets using two non-parametric tests: the Kolmogorov-Smirnov (KS) test, which compares overall distribution shape and location, and the Mann-Whitney U test, which specifically tests for a location/rank shift.

- **KS test**: statistic = 0.339, p-value = 7.77 × 10⁻⁶⁶
- **Mann-Whitney U test**: statistic = 509419.5, p-value = 3.89 × 10⁻⁶⁴

Both p-values are far below any conventional significance threshold, so we reject H₀ in both cases: the radius distributions differ significantly between short- and long-period planets, both in shape and location. This is also a practically large effect (median 1.67 R⊕ vs 2.34 R⊕), not merely a large-sample artifact, and is consistent with period-dependent atmospheric mass-loss (photoevaporation / core-powered mass-loss) shrinking close-in planets below the gap more often.

In [4]:
df['period_group3'] = pd.cut(df['pl_orbper'], bins=[0, 5, 20, df['pl_orbper'].max()], labels=['short', 'medium', 'long'])
print(df['period_group3'].value_counts())

groups = [df.loc[df['period_group3'] == g, 'pl_rade'] for g in ['short', 'medium', 'long']]
kw_stat, kw_p = kruskal(*groups)
print(f"\nKruskal-Wallis: statistic = {kw_stat:.4f}, p-value = {kw_p:.4e}")

period_group3
medium    1207
short      797
long       583
Name: count, dtype: int64

Kruskal-Wallis: statistic = 381.7936, p-value = 1.2433e-83


## Robustness check: radius across three period groups

As a robustness extension of the previous test, we re-split the sample into three period groups (short ≤5 days, medium 5-20 days, long >20 days) and applied the Kruskal-Wallis test, which extends the Mann-Whitney comparison to more than two groups.

- **Kruskal-Wallis**: statistic = 381.79, p-value = 1.24 × 10⁻⁸³

This confirms the same underlying finding as the two-group KS/Mann-Whitney tests above — it is not an independent discovery, but a robustness check showing that the period-radius relationship holds regardless of how finely the period axis is divided. The even smaller p-value reflects the finer grouping capturing a monotonic trend (radius increases with period) more sharply.

In [5]:
df['multi_planet_system'] = np.where(df['sy_pnum'] > 1, 'multi', 'single')

contingency = pd.crosstab(df['multi_planet_system'], df['above_gap'])
print(contingency)

chi2_stat, chi2_p, dof, expected = chi2_contingency(contingency)
print(f"\nChi-square statistic = {chi2_stat:.4f}, p-value = {chi2_p:.4e}, dof = {dof}")
print("\nExpected counts:")
print(pd.DataFrame(expected, index=contingency.index, columns=contingency.columns))

above_gap              0    1
multi_planet_system          
multi                535  710
single               482  860

Chi-square statistic = 13.1811, p-value = 2.8279e-04, dof = 1

Expected counts:
above_gap                     0           1
multi_planet_system                        
multi                489.433707  755.566293
single               527.566293  814.433707


## Association between multi-planet systems and gap position

A chi-square test of independence tested whether membership in a multi-planet system is associated with a planet's position relative to the radius gap.

- **Chi-square statistic** = 13.18, **p-value** = 2.83 × 10⁻⁴, dof = 1

All expected cell counts exceed 5, satisfying the test's validity assumption. We reject H₀ (independence) at the 0.05 level: there is a statistically significant association between multi-planet system membership and gap position. Planets in multi-planet systems are somewhat over-represented below the gap (535 observed vs. 489.4 expected) and under-represented above it (710 observed vs. 755.6 expected) — consistent with the "peas in a pod" effect, where planets in the same system tend toward similar (and often smaller) sizes.

In [6]:
df['transit_vs_other'] = np.where(df['discoverymethod'] == 'Transit', 'Transit', 'Other')

contingency2 = pd.crosstab(df['transit_vs_other'], df['above_gap'])
print(contingency2)

odds_ratio, fisher_p = fisher_exact(contingency2)
print(f"\nFisher's exact test: odds ratio = {odds_ratio:.4f}, p-value = {fisher_p:.4f}")

above_gap            0     1
transit_vs_other            
Other               11    14
Transit           1006  1556

Fisher's exact test: odds ratio = 1.2153, p-value = 0.6827


## Association between detection method and gap position

Since discovery method counts are heavily imbalanced after filtering (2562 Transit vs. 25 combined Radial Velocity/TTV), a chi-square test would be unreliable here (expected cell counts would be too small). Fisher's exact test, designed for small-sample 2×2 tables, was used instead, comparing Transit vs. all other detection methods combined.

- **Fisher's exact test**: odds ratio = 1.22, p-value = 0.68

This result is not statistically significant. We do not reject H₀: there is no evidence of an association between detection method and gap position. This is likely due to limited statistical power from the very small "Other" group (n=25) rather than genuine equivalence — the result should be read as inconclusive, not as proof of no association.

In [7]:
predictors_to_test = ['pl_insol', 'st_mass', 'st_met', 'pl_orbeccen']

raw_p = {}
for col in predictors_to_test:
    group0 = df.loc[df['above_gap'] == 0, col].dropna()
    group1 = df.loc[df['above_gap'] == 1, col].dropna()
    stat, p = mannwhitneyu(group0, group1, alternative='two-sided')
    raw_p[col] = p
    print(f"{col}: n0={len(group0)}, n1={len(group1)}, U={stat:.1f}, p={p:.4e}")

# Multiple comparisons: this section runs several tests on the same sample.
from statsmodels.stats.multitest import multipletests
cols = list(raw_p)
_, p_holm, _, _ = multipletests([raw_p[c] for c in cols], method='holm')
print("\nHolm-adjusted p-values:")
for c, pa in zip(cols, p_holm):
    print(f"  {c}: {pa:.4e}{'  (still significant at 0.05)' if pa < 0.05 else '  (NOT significant at 0.05)'}")

print()
print(df.groupby('above_gap')[predictors_to_test].median())

pl_insol: n0=947, n1=1459, U=931110.5, p=3.2094e-47
st_mass: n0=1017, n1=1570, U=686193.5, p=1.5041e-09
st_met: n0=979, n1=1466, U=661783.5, p=1.0965e-03
pl_orbeccen: n0=845, n1=1226, U=468563.5, p=3.7928e-07

Holm-adjusted p-values:
  pl_insol: 1.2838e-46  (still significant at 0.05)
  st_mass: 4.5122e-09  (still significant at 0.05)
  st_met: 1.0965e-03  (still significant at 0.05)
  pl_orbeccen: 7.5856e-07  (still significant at 0.05)



           pl_insol  st_mass  st_met  pl_orbeccen
above_gap                                        
0             166.0    0.854   -0.01          0.0
1              47.3    0.900    0.00          0.0


## Group differences in the candidate predictors

We tested whether each candidate predictor differs between above-gap and under-gap planets using Mann-Whitney U tests.

Two caveats on how these should be read:

- **`pl_orbeccen` was *excluded* from the Phase 6 models** (20% missing), so it is a candidate predictor tested here, not one the models identified as important.
- These tests are **not independent confirmation** of the Phase 6 feature importances: same sample, same variables, same labels. They restate the marginal signal the models already used, in classical form.

- **pl_insol**: p = 3.21 × 10⁻⁴⁷ — under-gap planets have much higher insolation (median 166.0 vs. 47.3), consistent with closer-in planets losing more atmosphere to irradiation.
- **st_mass**: p = 1.50 × 10⁻⁹ — above-gap planets orbit slightly more massive stars (median 0.900 vs. 0.854 M☉).
- **st_met**: p = 1.10 × 10⁻³ — above-gap planets orbit slightly more metal-rich stars (median 0.00 vs. -0.01 dex), a smaller but still significant effect.
- **pl_orbeccen**: p = 3.79 × 10⁻⁷ — significant, despite both groups sharing an identical median (0.0). This reflects a difference in the upper tail of the distribution rather than the center: many planets default to an assumed circular orbit (eccentricity = 0) when not well constrained, so the Mann-Whitney test is picking up a difference in rank distribution, not a shift in typical value.

All four differences are statistically significant and remain so after Holm correction. They are consistent with the Phase 6 feature-importance patterns, but restate rather than independently corroborate them.

# Phase 7 Summary: Classical Hypothesis Testing

This phase formally tested, using classical non-parametric and categorical statistics, the group differences that Phase 6's classification models identified informally through coefficients and feature importance.

**Period → radius relationship** (tested three ways for robustness): KS test (p = 7.77 × 10⁻⁶⁶), Mann-Whitney U (p = 3.89 × 10⁻⁶⁴), and Kruskal-Wallis across three period groups (p = 1.24 × 10⁻⁸³) all confirm the same finding — radius depends strongly on orbital period — with increasing significance as the grouping becomes finer. These three tests represent one robustly confirmed result, not three independent findings.

**Categorical associations with gap position**: A chi-square test found a significant association between multi-planet system membership and gap position (p = 2.83 × 10⁻⁴), consistent with the "peas in a pod" effect. A Fisher's exact test found no significant association between detection method and gap position (p = 0.68), though this is likely underpowered given the small non-Transit sample (n=25) rather than genuine evidence of no effect.

**Predictor group differences**: Mann-Whitney tests on `pl_insol`, `st_mass`, `st_met`, and `pl_orbeccen` were all significant (p < 0.01 in every case, and after Holm correction). Note `pl_orbeccen` was not in the Phase 6 models, and these tests use the same data and labels as those models, so they restate rather than independently confirm the feature-importance result. A notable methodological nuance: `pl_orbeccen` showed identical medians across groups despite a significant test, illustrating that Mann-Whitney detects differences in the overall rank distribution rather than strictly the median.

**Overall conclusion:** Every hypothesis test in this phase is consistent with the descriptive and model-based findings from Phases 4-6: the radius gap is closely tied to orbital period and insolation (consistent with atmospheric mass-loss theory), with secondary associations to system architecture and stellar properties.